__Prerequisites:__

* Access to the [`bdcmsg22/QC_metabolomics`](https://app.terra.bio/#workspaces/bdcmsg22/QC_metabolomics) workspace where the "`PhenoUsmanSIDNO`" file is stored.
* Access to the [`mgb-KEW-K01-GCP/gxpa-lcms-mediation`](https://app.terra.bio/#workspaces/mgb-KEW-K01-GCP/gxpa-lcms-mediation) workspace where other phenotype files are stored.
* Access to the [`manning-lab-2024-2025/manning-lab-2024-2025-topmed-analysis`](https://app.terra.bio/#workspaces/manning-lab-2024-2025/manning-lab-2024-2025-topmed-analysis) workspace where the kinship matrix is stored.

# Setup

In [ ]:
pkgs <- c('data.table','Matrix','readxl')
lapply(pkgs, \(pkg) { if(!require(pkg, character.only=T)) {install.packages(pkg);require(pkg)}}) |> invisible()

options(datatable.na.strings=c('NA',''))

if(!file.exists('data/raw/phenotypes/')) {
  dir.create('data/raw/phenotypes',rec=T)
  system('gcloud storage cp gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/data/raw/phenotypes/* data/raw/phenotypes')
  system('gcloud storage cp gs://fc-secure-4b3e979d-ba8b-43c4-a5ec-b41ab42ce606/MesaMetabolomics_PilotX01_2023_02_16/MESA_PhenoUsmanSIDNO_20230214.txt data/raw/phenotypes')
}
system('gcloud storage cp gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/data/raw/genomics/dosages.csv data/raw/genomics')

winsorizeBySd <- function(x, SDs=5) {
  lower_bound <- mean(x,na.rm=T) - SDs*sd(x,na.rm=T)
  upper_bound <- mean(x,na.rm=T) + SDs*sd(x,na.rm=T)
  print(paste(sum(x<lower_bound,na.rm=T), 'values winsorized at the lower bound.'))
  print(paste(sum(x>upper_bound,na.rm=T), 'values winsorized at the upper bound.'))
  x[x<lower_bound] <- lower_bound
  x[x>upper_bound] <- upper_bound
  return(x)
}

In [ ]:
# Merge which coalesces columns with duplicated names along the way.
# Note: when coalescing same-name cols of x and y, x's values will be chosen first.
mergecoalesce <- \(x,y, by=NULL, by.x=NULL, by.y=NULL, ..., force_coalesce=F) {
  if(!is.null(by)) by.x <- by.y <- by
  setkeyv(x,by)
  setkeyv(y,by)
  merged <- merge(x,y, by.x=by.x, by.y=by.y, ...)

  nm_dups <- intersect(names(x),names(y)) |> setdiff(c(by.x,by.y))

  for(nm in nm_dups) {
    nms_w_suffix <- paste0(nm, c('.x','.y'))

    classes <- merged[, sapply(.SD,class), .SDcols=nms_w_suffix]
    if(length(unique(classes))!=1L) { # Danger, cols' classes don't match
      if(!force_coalesce) {
        warning('Not coalescing these columns because their class doesn\'t match:\n', paste(capture.output(print(classes)), collapse='\n'))
        next
      } else { # Force coalescing anyway (cast to character for maximum compatibility)
        merged[, (nms_w_suffix) := lapply(.SD, as.character), .SDcols = nms_w_suffix]
    }}

    merged <- merged[
      ][, (nm) := fcoalesce(.SD), .SDcols= nms_w_suffix 
      ][, .SD                   , .SDcols=!nms_w_suffix]
  }

  merged
}

# Detects cols with duplicated names in dt, and coalesces them.
coalesce_samenames <- \(dt, force_coalesce=F) {
  nms <- unique(colnames(dt)[duplicated(colnames(dt))])
  ns  <- sapply(nms,\(nm) sum(nm==colnames(dt)))
  exact_nms <- paste0('^',nms,'$')

  for(i in seq_along(nms)) {
    nm <- nms[i]
    n  <- ns [i]
    exact_nm <- exact_nms[i]

    classes <- dt[, sapply(.SD,class), .SDcols=patterns(exact_nm)]
    if(length(unique(classes))!=1L) { # Danger, cols' classes don't match
      if(!force_coalesce) { 
        warning('Not coalescing these columns because their class doesn\'t match:\n', paste(capture.output(print(classes)), collapse='\n'))
        next
      } else { # Force coalescing despite type mismatch (cast to character for maximum compatibility)
        cols_as.char <- dt[, lapply(.SD,as.character), .SDcols=patterns(exact_nm)]
        coalesced_col <- fcoalesce(cols_as.char)
      }
    } else { # All is well, cols' classes match
      coalesced_col <- dt[, fcoalesce(.SD), .SDcols=patterns(exact_nm)]
    }

    for(j in 1:n) dt[, (nm) := NULL]
    dt[, (nm) := coalesced_col]
  }
  dt
}

# Join all datasets

In [ ]:
# Frist, create a "base" dt containing all the usable sample IDs.
# That is, TOM_Ids with metabolomics info, and mesa_ids with PA info. Note that one mesa_id may correspond to multiple TOM_Ids.
# Then, we can repeatedly merge into that base with impunity (using all.x=T), without fear of accidently dropping any info.
data <-
mergecoalesce(by='mesa_id', all=T, force_coalesce=T,
  fread('data/raw/phenotypes/MESA_PhenoUsmanSIDNO_20230214.txt')[, mesa_id := sidno],
  fread('data/derived/phenotypes/mesa_picsure_variables.csv')   [, mesa_id := `\\phs000209\\pht001116\\phv00084441\\sidno\\`]
  )[,mesa_id:=as.character(mesa_id)][,NONE:=NA]      |> # Add an NA "NONE" column, useful to fill variables with no measurements for a particular MESA exam.
  setnames(\(nm)  sub('.*\\\\(.*)\\\\$', '\\1', nm)) |> # Fixes some names e.g. '\phs000209\phv00084442\age1c\' -> 'age1c'
  coalesce_samenames(force_coalesce=T)               |>
  melt(
    variable.name = 'exam',
    measure.vars = list(
      TOM_Id         =c(      'tomid1',      'tomid2',        'tomid3',       'tomid4',       'tomid5' ),
      TOP_Id         =c(      'topid1',      'topid2',        'topid3',       'topid4',       'topid5' ),
      TOE_Id         =c(      'toeid1',      'toeid2',        'toeid3',       'toeid4',       'toeid5' ),
      hdl            =c(        'hdl1',        'hdl2',          'hdl3',         'hdl4',         'hdl5' ),
      tg             =c(       'trig1',       'trig2',         'trig3',        'trig4',        'trig5' ),
      season         =c(     'season1',     'season2',       'season3',      'season4',      'season5' ),
      month          =c(      'month1',      'month2',        'month3',       'month4',       'month5' ),
      ses_score      =c(    'F1_PC2_1',    'F1_PC2_2',      'F1_PC2_3',     'F1_PC2_4',     'F1_PC2_5' ),
      income         =c(     'income1',     'income2',       'income3',         'NONE',      'income5' ),
      ahei_score     =c( 'ahei_2010_1',         'NONE',         'NONE',         'NONE',         'NONE' ),
      dash_score     =c('dash_sodium1',         'NONE',         'NONE',         'NONE',         'NONE' ),
      drinks_per_week=c(      'alcwk1c',        'NONE',         'NONE',         'NONE',         'NONE' ),
      age            =c(        'age1c',       'age2c',         'age3c',        'age4c',        'age5c'),
      bmi            =c(        'bmi1c',       'bmi2c',         'bmi3c',        'bmi4c',        'bmi5c'),
      smoking        =c(        'cig1c',       'cig2c',         'cig3c',        'cig4c',        'cig5c'),
      site           =c(       'site1c',      'site2c',        'site3c',       'site4c',       'site5c'),
      hb             =c(         'NONE',      'hba1c2',          'NONE',         'NONE',       'hba1c5'),
      fg             =c(     'glucos1c',    'glucos2c',      'glucos3c',     'glucos4c',     'glucose5'),
      insulin        =c(      'insln1c',     'insln2c',       'insln3c',      'insln4c',      'insln5c'),
          pa         =c(     'exercm1c',    'exercm2c',      'exercm3c',         'NONE',     'exercm5c'),
      mod_pa         =c(      'pamcm1c',     'pamcm2c',       'pamcm3c',         'NONE',      'pamcm5c'),
        mvpa         =c(     'pamvcm1c',    'pamvcm2c',      'pamvcm3c',         'NONE',     'pamvcm5c'),
      vig_pa         =c(      'pavcm1c',     'pavcm2c',       'pavcm3c',         'NONE',      'pavcm5c'))) |> suppressWarnings()
data <- data[
  ][ !(duplicated(mesa_id) & is.na(TOM_Id)) # Only keep 1 row per TOM_Id. Or, 1 row if a mesa_id has no TOM_Ids.
  ][, NWD_Id := fcoalesce(nwdid2,nwdid1)    # Merge NWD_Id (genotype ID) columns. Prefer the exam 2 genotype (more recent).
]

data <-
Reduce(\(x,l) mergecoalesce(x, l$y, by=l$by, force_coalesce=T, all.x=T),
  list(                x=data,
    list(by='mesa_id', y=fread('data/raw/phenotypes/nmr_metabolites.csv')                 [, mesa_id := sub('phs000209.v13_', '', `\\_Parent Study Accession with Subject ID\\`)]),
    list(by='mesa_id', y=fread('data/raw/phenotypes/mesa5_phenos_diet.csv')               [, mesa_id := sub('phs000209.v13_', '', `\\_Parent Study Accession with Subject ID\\`)]),
    list(by='mesa_id', y=fread('data/raw/phenotypes/mesa5_phenos_basic.csv')              [, mesa_id := sub('phs000209.v13_', '', `\\_Parent Study Accession with Subject ID\\`)]),
    list(by='mesa_id', y=fread('data/raw/phenotypes/id_match_file.csv')                   [, mesa_id := as.character(Cohort_Specific_Id)][!duplicated(mesa_id)]                  ), # Duplicated mesa_ids due to TOR id replicates. We aren't using RNAseq data so the duplicate we drop doesn't matter.
    list(by='mesa_id', y=fread('data/raw/phenotypes/freeze9b_sample_annot_2020-08-20.txt')[, mesa_id := subject_id][!duplicated(mesa_id)]                                        ),
    list(by='mesa_id', y=fread('data/raw/phenotypes/SHARe_AncilMesaNMR_LP4_DS.txt')       [, mesa_id := as.character(subject_id)]                                                ),
    list(by= 'NWD_Id', y=fread('data/raw/genomics/dosages.csv')                                                                                                                  ),
    list(by= 'NWD_Id', y=fread('data/raw/phenotypes/freeze9_pcair_results.tsv')           [, NWD_Id := sample.id]                                                                ),
    list(by= 'TOM_Id', y=fread('data/derived/metabolomics/sample_info-mesa.csv')          [, TOM_Id := sample_id]                                                                ),
    list(by= 'TOM_Id', y=fread('data/derived/metabolomics/QCd/merged_QCd-mesa.csv')                                                                                              )) # Merge metabolomics last b/c it adds tons of cols
)

# Rename columns

In [ ]:
data <- data |>
  setnames(\(nm)  sub('.*\\\\(.*)\\\\$', '\\1', nm)) |> # Fixes some names e.g. '\phs000209\phv00084442\age1c\' -> 'age1c'
  setnames(\(nm) gsub(      '\\\\'     ,  '',   nm)) |> # Rm remaining backslashes
  setnames(\(nm)  sub('^PC'            , 'gPC', nm)) |> # 'PC#' -> 'gPC#' To distinguish genetic PCs from metabolite PCs we'll add later
  coalesce_samenames(force_coalesce=T)                  # Coalesce any columns whose names are now duplicated after renaming stuff
data <- data[, `:=`(                                    # Manually pretty up some names
   # SHARe...txt      #nmr...csv
   HDL_C=nhdlc1,      HDL_C_lp3=nhdlc31c,
   HDL_P=chdlp1,      HDL_P_lp3= hdlp31c,
   HDL_size=hdlz1,    HDL_size_lp3=hz31,
   L_HDL_P=l_chdlp1,  L_HDL_P_lp3=hl31,
   M_HDL_P=m_chdlp1,  M_HDL_P_lp3=hm31,
   S_HDL_P=s_chdlp1,  S_HDL_P_lp3=hs31,
   H1P=h1p1,
   H2P=h2p1,  #id_match_file.csv
   H3P=h3p1,  prop_African=African,
   H4P=h4p1,  prop_American=American,
   H5P=h5p1,  prop_Eas_Asian=East_Asian,
   H6P=h6p1,  prop_European=European,
   H7P=h7p1
)]

## Winsorization, imputation, transformation

In [ ]:
winsor      <- \(v, bounds=quantile(v,c(0,0.9),na.rm=T)) pmin(pmax(v,bounds[1]),bounds[2])
na_outliers <- \(v, bounds=quantile(v,c(0,0.9),na.rm=T)) fifelse(v<bounds[1] | v>bounds[2], NA, v)
nafill2 <- \(x) c(NA,x[!is.na(x)])[cumsum(!is.na(x))+1] # nafill(type='locf') but can handle non-numeric

setorder(data,exam)
data <- data[
  # Winsorization
  ][, .SDcols=c('pa','mvpa','mod_pa','vig_pa'),
      names(.SD) := lapply(.SD, \(x) winsorizeBySd(x/60))
  ][, .SDcols = c('bmi','ahei_score','dash_score','ses_score','drinks_per_week'),
      names(.SD) := lapply(.SD, \(x) winsorizeBySd(x   ))

  # (Median) imputation of covariates if not present in any exam (i.e. not present in exam 1)
  # (Making sure to take the median of the original set of non-missing values, NOT recalculating the median after each imputation!)
  ][, {  age_median <<- median(            age, na.rm=T);
         bmi_median <<- median(            bmi, na.rm=T);
         ses_median <<- median(      ses_score, na.rm=T);
        ahei_median <<- median(     ahei_score, na.rm=T);
        dash_median <<- median(     dash_score, na.rm=T);
       drink_median <<- median(drinks_per_week, na.rm=T); .SD }

  ][ exam==1 & is.na(            age),             age :=   age_median # Technically we have the information to impute age more accurately but I am too lazy and it's only for 117 samples
  ][ exam==1 & is.na(            bmi),             bmi :=   bmi_median
  ][ exam==1 & is.na(      ses_score),       ses_score :=   ses_median
  ][ exam==1 & is.na(     ahei_score),      ahei_score :=  ahei_median
  ][ exam==1 & is.na(     dash_score),      dash_score :=  dash_median
  ][ exam==1 & is.na(drinks_per_week), drinks_per_week := drink_median

  ][ exam==1 & is.na(smoking) | smoking=='', smoking := 'NEVER'
  ][ exam==1 & is.na(income)  |  income=='', income  := 'Missing'
  ][ exam==1 & is.na(season)  |  season=='', season  := 'Unknown' # All the same samples are missing season/month/site. 117 samps '', 21 samps NA. Should these be kept as 'Unknown' or discoarded? Only 2 have genotype data so doesn't really matter.
  ][ exam==1 & is.na(month)   |   month=='', month   := 'Unknown'
  ][ exam==1 & is.na(site)    |    site=='', site    := 'Unknown'

  # Carry-over-from-previous-exam imputation (this is why we setorder()'d by exam before)
  ][, .SDcols=c('hdl','fg','hb', 'pa','mvpa','mod_pa','vig_pa',
                'age','bmi','sex',
                'ahei_score','dash_score','ses_score','drinks_per_week',
                'smoking','income','site','month','season'),
      names(.SD) := lapply(.SD, nafill2), # We preiviously setorder()'d by exam so this will fill exams in starting from the first.
      by=mesa_id

  # Definitions
  ][, pa_bin    := pa > 3.75
  ][, hdl_log   := log(hdl)
  ][,  tg_log   := log(tg)
  ][, race      := fifelse(race1c==1, 'WHITE, CAUCASIAN',
                   fifelse(race1c==2, 'CHINESE AMERICAN',
                   fifelse(race1c==3, 'BLACK, AFRICAN-AMERICAN',
                   fifelse(race1c==4, 'HISPANIC', as.character(race1c))))) |> as.factor()
  ][, smoking   := fifelse(smoking==0, 'NEVER',
                   fifelse(smoking==1, 'FORMER',
                   fifelse(smoking==2, 'CURRENT', smoking))) |> as.factor()
  ][, income    := income |>
                   gsub(pattern='$', replacement='', fixed=T) |> # Harmonize inconsistent formatting ($1,000 / $1000 / 1000).
                   gsub(pattern=',', replacement='', fixed=T) |>
                   as.factor() # Treating income as unordered factor for simplicity, and b/c it might not have a monotonic relationship w/ outcomes. Update: we end up not using income as a covariate anyway.
  ][, education := factor(educ1,  levels=c('NO_SCHOOLING','GRADES 1-8','GRADES 9-11','COMPLETED HIGH SCHOOL/GED','SOME COLLEGE BUT NO DEGREE','ASSOCIATE DEGREE','TECHNICAL SCHOOL CERTIFICATE',"BACHELOR'S DEGREE",'GRADUATE OR PROFESSIONAL SCHOOL'))
  ][, season    := as.factor(season)
  ][, month     := as.factor(month)
  ][, site      := as.factor(site)

  ][, sex := fcoalesce(as.character(gender1),Sex,SEX) |>
        sub(pattern='0|^F.*', replacement='FEMALE', ignore.case=T) |>
        sub(pattern='1|^M.*', replacement=  'MALE', ignore.case=T) |>
        factor(levels=c('FEMALE','MALE'))
      # There is an existing `sex` col in freeze9b_sample_annot_2020-08-20.txt,
      #   but it disagrees with the other `Sex`, `SEX`, and `gender1` cols. Ignoring it.

  # Transformed PA variables
  ][,     pa_trunc := na_outliers(    pa)
  ][,   mvpa_trunc := na_outliers(  mvpa)
  ][, mod_pa_trunc := na_outliers(mod_pa)
  ][, vig_pa_trunc := na_outliers(vig_pa)
  ][,     pa_wins  :=      winsor(    pa)
  ][,   mvpa_wins  :=      winsor(  mvpa)
  ][, mod_pa_wins  :=      winsor(mod_pa)
  ][, vig_pa_wins  :=      winsor(vig_pa)
  ][,     pa_log   :=         log(    pa+1)
  ][,   mvpa_log   :=         log(  mvpa+1)
  ][, mod_pa_log   :=         log(mod_pa+1)
  ][, vig_pa_log   :=         log(vig_pa+1)
]

# Append metabolomics PCs
We waited to calculate PCs until now because we only want to do PCA using samples we will use in the analysis; and now we are finally done filtering samples.\
We performed half-minimum imputation in metabolomics QC, but NAs are reintroduced when merging the metabolomics methods CP/CN/HP/AN, since not all samples got profiled with all the LC-MS methods.\
PCA is intolerant to missing values, so we remove samples who have any missing values. Perhaps a more sophisticated imputation strategy would be better, but the mPCs are only used for exploratory analysis anyways.

In [ ]:
met_nms <- names(fread('data/derived/metabolomics/QCd/merged_QCd-mesa.csv',nrows=0,drop=1))
met_mtx <- data[
  ][ rowSums(is.na(data[,..met_nms]))==0 # Only samples with data for ALL metabolomics methods CP/CN/HP/AN, because PCA can't handle NAs
  ][ exam==1 # TODO consider a way to include all exams' met data into the PCs? mPCs only used for rough exploration to probably not important, but idk.
  ][, c('TOM_Id', ..met_nms)
] |> as.matrix(rownames=1)

pca <- prcomp(met_mtx, center=T, scale=T)
pcs <- as.data.table(pca$x[,1:20], keep.rownames=T)
setnames(pcs, c('TOM_Id',paste0('mPC', 1:20)))
data <- pcs[data, on='TOM_Id']

screeplot(pca,npcs=20)

# Remap kinship matrix ids

In [ ]:
dir.create('data/raw/genomics',rec=T)
dir.create('data/derived/genomics',rec=T)

if(!file.exists('data/raw/genomics/km.Rdata')) system('gcloud storage cp gs://fc-secure-c5905035-bf75-471b-8fa8-fb8a7f83c4a9/submissions/e81b3733-44cb-4c17-87db-2aee99a0491a/fetch_dbgap/d64cc7fd-47b7-4352-985f-3c3951b9c3cf/call-decrypt/glob-b3ddbe8d141590fbae0db21546878fa2/km.Rdata data/raw/genomics')
load('data/raw/genomics/km.Rdata')
km <- km[rownames(km) %in% data$NWD_Id,
         colnames(km) %in% data$NWD_Id]

dimnames(km) <- list(data[match(rownames(km), NWD_Id), mesa_id],
                     data[match(colnames(km), NWD_Id), mesa_id])

save(km, file='data/derived/genomics/mesa_km.RData')

## Write

In [ ]:
fwrite(data, 'data/derived/analysis_df-mesa.csv')

In [ ]:
system('gcloud storage cp data/derived/analysis_df-mesa.csv   gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/data/derived/')
system('gcloud storage cp data/derived/genomics/mesa_km.RData gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/data/derived/genomics/')